# PaperHub Jupyter Quickstart

This notebook walks through running PaperHub from a fresh checkout.

Flow:

1. Make sure the notebook is running from the repository root.
2. Install the package in editable mode.
3. Provide an API key safely.
4. Run setup, import, launcher, and quick test checks.
5. Check the HuggingFace Daily Papers metadata path without calling an LLM.
6. Run a real summary call if an API key is ready.
7. Explore `PaperHub.run`, date formats, output formatting, cache behavior, and async usage.

Note: Do not write API keys into the notebook as plain text. The `getpass` cell below reads the key as hidden input and keeps it only as an environment variable for this kernel session.


## 1. Find The Repository Root

This cell finds the project root containing `pyproject.toml` and `src/paperhub`, then moves the working directory there no matter where the notebook was opened from.


In [ ]:
import os
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "paperhub").exists():
            return candidate
    raise RuntimeError(
        "Could not find the PaperHub repository root. Open the notebook inside the project."
    )


REPO_ROOT = find_repo_root(Path.cwd().resolve())
os.chdir(REPO_ROOT)
print(f"Working directory: {REPO_ROOT}")

## 2. Install The Package

This cell installs PaperHub in editable mode, so changes under `src/paperhub` are picked up without reinstalling the package.

The default provider is `openai`, and the default OpenAI model is `gpt-5.4-mini` with `xhigh` reasoning effort. If you want Anthropic or Google, change `PROVIDER` below. If `MODEL` is left as `None`, PaperHub uses the selected provider's configured default model.


In [ ]:
import subprocess
import sys

# Options: "openai", "anthropic", "google"
PROVIDER = "openai"

# Output language: "en" by default, or "tr" for Turkish.
LANGUAGE = "en"

# You do not have to set a model. Leave None to use the selected provider's default.
# Example override: MODEL = "gpt-5.4-mini"
MODEL = None

DEFAULT_MODEL_BY_PROVIDER = {
    "anthropic": "claude-haiku-4-5-20251001",
    "openai": "gpt-5.4-mini",
    "google": "gemini-3-flash-preview",
}

subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", ".[dev]"])

if PROVIDER == "anthropic":
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", ".[anthropic]"])
elif PROVIDER == "google":
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", ".[google]"])

print(f"Setup complete: provider={PROVIDER}, language={LANGUAGE}, model_override={MODEL or 'none'}")

## 3. Provide An API Key

There are two safe options.

### Option A: Hidden input for this notebook session

The cell below first checks the `.env` file in the repository root. If the selected provider's API key is present there, it will be copied into an environment variable for this kernel session.

If the key is not in `.env`, `getpass` asks for it as hidden input. The key is not saved into the notebook file; it only lives in the active kernel session.

### Option B: `.env` file

In a terminal or editor, copy `.env.example` to `.env` at the project root:

```bash
cp .env.example .env
```

Then fill the relevant line:

- OpenAI: `OPENAI_API_KEY=...`
- Anthropic: `ANTHROPIC_API_KEY=...`
- Google: `GOOGLE_API_KEY=...`

The `.env` file should not be committed; `.gitignore` already excludes it.


In [ ]:
import os
from getpass import getpass

from dotenv import dotenv_values

API_KEY_ENV_BY_PROVIDER = {
    "anthropic": "ANTHROPIC_API_KEY",
    "openai": "OPENAI_API_KEY",
    "google": "GOOGLE_API_KEY",
}

api_key_env = API_KEY_ENV_BY_PROVIDER[PROVIDER]
env_path = REPO_ROOT / ".env"
env_values = dotenv_values(env_path) if env_path.exists() else {}
key_source = "environment"

env_file_key = (env_values.get(api_key_env) or "").strip()
if not os.environ.get(api_key_env) and env_file_key:
    os.environ[api_key_env] = env_file_key
    key_source = ".env"
elif not os.environ.get(api_key_env):
    value = getpass(f"Enter {api_key_env} (hidden): ").strip()
    if value:
        os.environ[api_key_env] = value
        key_source = "getpass"
    else:
        key_source = "missing"

MODEL_ENV_BY_PROVIDER = {
    "anthropic": "PAPERHUB_ANTHROPIC_MODEL",
    "openai": "PAPERHUB_OPENAI_MODEL",
    "google": "PAPERHUB_GOOGLE_MODEL",
}

provider_model_env = MODEL_ENV_BY_PROVIDER[PROVIDER]
env_file_provider_model = (env_values.get(provider_model_env) or "").strip()
env_file_global_model = (env_values.get("PAPERHUB_MODEL") or "").strip()

if MODEL is None and env_file_provider_model:
    os.environ[provider_model_env] = env_file_provider_model
elif MODEL is None:
    os.environ.setdefault(provider_model_env, DEFAULT_MODEL_BY_PROVIDER[PROVIDER])

os.environ["PAPERHUB_PROVIDER"] = PROVIDER
if MODEL is None:
    os.environ.pop("PAPERHUB_MODEL", None)
else:
    os.environ["PAPERHUB_MODEL"] = MODEL
os.environ.setdefault("PAPERHUB_CONCURRENCY", "2")
os.environ.setdefault("PAPERHUB_MAX_PDF_CHARS", "60000")

effective_model_hint = (
    MODEL or os.environ.get(provider_model_env) or DEFAULT_MODEL_BY_PROVIDER[PROVIDER]
)
if MODEL is None and env_file_global_model:
    print(
        "Note: PAPERHUB_MODEL is set in .env. If it matches the provider, PaperHub uses it; "
        "otherwise it falls back to the selected provider's default model."
    )

print(f"Provider: {PROVIDER}")
print(f"Output language: {LANGUAGE}")
print(f"Model override: {MODEL or 'none'}")
print(f"Provider default model candidate: {effective_model_hint}")
print(f"{api_key_env}: {'ready' if os.environ.get(api_key_env) else 'missing'} ({key_source})")

## 4. Readiness Check

This section checks that the package imports, settings load, and quick unit tests pass. These checks do not make real LLM calls.


In [ ]:
import httpx

import paperhub
from paperhub import PaperHub, RunRequest, load_settings, render_markdown, render_plain
from paperhub.dates import resolve_range
from paperhub.fetchers import HFAPIFetcher, HFHtmlFetcher, papers_in_range

settings = load_settings()

print("paperhub version:", paperhub.__version__)
print("package path:", paperhub.__file__)
print("settings provider:", settings.paperhub_provider)
print("settings global model override:", settings.paperhub_model or "none")
print("effective provider model:", settings.model_for_provider(PROVIDER))
print("cache dir:", settings.cache_dir())

request = RunRequest(period="month", year=2026, month=5, top_n=3, language=LANGUAGE)
print("programmatic request example:", request)

assert hasattr(paperhub, "PaperHub")
assert request.period == "month"
print("Import and programmatic request check passed.")

In [ ]:
quick_tests = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "tests/test_public_api.py",
        "tests/test_nl_parser.py",
        "tests/test_dates.py",
    ],
    text=True,
    capture_output=True,
)

print(quick_tests.stdout)
if quick_tests.stderr:
    print(quick_tests.stderr)
assert quick_tests.returncode == 0, "One of the quick tests failed. Review the output above."
print("Quick test check passed.")

## 5. HuggingFace Metadata Check

This smoke test fetches only HuggingFace Daily Papers metadata. It does not download PDFs or call an LLM.

The goal is to confirm that the internet connection and HuggingFace fetch pipeline are ready.

The cell tries several candidate dates with the programmatic metadata path, then stores the first matching arguments as `LIVE_RUN_KWARGS`. The real summary cell below uses those arguments.


In [ ]:
CANDIDATE_RUNS = [
    {
        "label": "2024-05-01",
        "kwargs": {"period": "day", "year": 2024, "month": 5, "day": 1, "top_n": 1},
    },
    {
        "label": "2024-10-01",
        "kwargs": {"period": "day", "year": 2024, "month": 10, "day": 1, "top_n": 1},
    },
    {
        "label": "May 2026",
        "kwargs": {"period": "month", "year": 2026, "month": 5, "top_n": 1},
    },
]

selected_run = None

for candidate in CANDIDATE_RUNS:
    request = RunRequest(**candidate["kwargs"], language=LANGUAGE)
    start_date, end_date = resolve_range(request)
    print(f"\n--- Trying: {candidate['label']} ({start_date} to {end_date})")
    async with httpx.AsyncClient(
        follow_redirects=True, timeout=settings.paperhub_request_timeout_s
    ) as client:
        primary = HFAPIFetcher(client)
        fallback = HFHtmlFetcher(client)
        papers = await papers_in_range(
            primary, start_date, end_date, request.top_n, fallback=fallback
        )
    print(f"Found {len(papers)} papers")
    if papers:
        selected_run = candidate
        break

if selected_run is None:
    raise RuntimeError(
        "Metadata smoke test did not find any papers. Check your internet connection or change CANDIDATE_RUNS."
    )

LIVE_RUN_KWARGS = selected_run["kwargs"]
LIVE_REQUEST = RunRequest(**LIVE_RUN_KWARGS, language=LANGUAGE)
print(
    f"\nSelected programmatic run for the real LLM call: {selected_run['label']} -> {LIVE_RUN_KWARGS}"
)

## 6. Create The PaperHub Object

`hub = PaperHub(...)` is the main entrypoint.

Important parameters:

- `provider`: `openai`, `anthropic`, or `google`.
- `model`: model id for the provider.
- `language`: `"en"` by default, or `"tr"` for Turkish summaries and labels.
- `concurrency`: how many paper agents may run at once. For a first run, `1` or `2` is easier to control.
- `max_pdf_chars`: character cap for PDF text sent to the model.
- `cache_dir`: optional custom cache folder.

If the same paper, model, and language are run again, the summary comes from cache and does not trigger another LLM charge.


In [ ]:
hub = PaperHub(
    provider=PROVIDER,
    model=MODEL,
    language=LANGUAGE,
    concurrency=2,
    max_pdf_chars=60000,
)

print("PaperHub ready.")
print("Provider:", hub.provider)
print("Model:", hub.model)
print("Language:", hub.language)
print("Concurrency:", hub.concurrency)
print("Cache root:", hub.cache.root)

## 7. Real Run: `hub.run(...)`

This cell runs the full pipeline:

1. Parse the date query.
2. Fetch HuggingFace Daily Papers metadata.
3. Download the relevant arXiv PDF.
4. Extract PDF text.
5. Run one `PaperAgent` per paper.
6. Ask the LLM for a structured English summary.
7. Render Markdown inside the notebook.
8. Return `list[PaperSummary]`.

Starting with `top 1` keeps cost and wait time low. Later you can increase it to `top 5`, `top 10`, and so on.


In [ ]:
assert os.environ.get(api_key_env), f"{api_key_env} is missing. Run the API key cell in step 3."

summaries = hub.run(**LIVE_RUN_KWARGS, display=True, language=LANGUAGE)

print(f"Returned summaries: {len(summaries)}")
for idx, summary in enumerate(summaries, start=1):
    status = "ERROR" if summary.error else "OK"
    print(f"{idx}. {status} | {summary.arxiv_id} | {summary.title}")
    if summary.error:
        print("   error:", summary.error)

## 8. Get Text Output

`hub.run(..., display=True)` renders Markdown in Jupyter. If you want plain text, use either of these paths:

1. `hub.run(..., display=False)` to get only the returned list.
2. `render_plain(summaries, request)` to create terminal-friendly text.

`render_markdown` creates a Markdown string for saving to a file or displaying elsewhere.


In [ ]:
request = LIVE_REQUEST

plain_text = render_plain(summaries, request)
markdown_text = render_markdown(summaries, request)

print("--- Plain text output ---")
print(plain_text[:4000])

print("\nMarkdown character count:", len(markdown_text))

## 9. Inspect Returned Objects

`hub.run(...)` returns a `list[PaperSummary]`. Each item contains:

- `arxiv_id`
- `title`
- `motivation`
- `method`
- `findings`
- `real_world_examples`
- `summary` — an English summary capped at 6000 characters by default
- `language`
- `pdf_chars`
- `model_used`
- `elapsed_s`
- `error` — populated for a paper-level failure; other papers keep running


In [ ]:
if summaries:
    first = summaries[0]
    data = first.model_dump()
    for key, value in data.items():
        if isinstance(value, str) and len(value) > 500:
            value = value[:500] + "..."
        print(f"{key}: {value}")
else:
    print("The summary list is empty. Change LIVE_RUN_KWARGS and try again.")

## 10. Date Formats

For Python code, use programmatic date arguments:

```python
hub.run(period="day", year=2026, month=5, day=1, top_n=5)
hub.run(period="week", year=2026, week=18, top_n=5)
hub.run(period="month", year=2026, month=5, top_n=10)
hub.run(period="year", year=2026, top_n=20)
hub.run(period="custom", start=date(2026, 4, 15), end=date(2026, 4, 30), top_n=15)
```

Natural-language date phrases are kept for the interactive launcher.


In [ ]:
example_requests = [
    {"period": "day", "year": 2026, "month": 5, "day": 1, "top_n": 5},
    {"period": "week", "year": 2026, "week": 18, "top_n": 5},
    {"period": "month", "year": 2026, "month": 5, "top_n": 10},
    {"period": "year", "year": 2026, "top_n": 20},
]

for kwargs in example_requests:
    print(kwargs, "->", RunRequest(**kwargs, language=LANGUAGE))

## 11. Additional Live Examples

This cell is disabled by default because every enabled example may trigger new LLM calls for new paper/model/language combinations.

Set `RUN_MORE_LIVE_EXAMPLES = True` to run the programmatic date examples for real.


In [ ]:
from datetime import date

RUN_MORE_LIVE_EXAMPLES = False

if RUN_MORE_LIVE_EXAMPLES:
    day_summaries = hub.run(period="day", year=2024, month=5, day=1, top_n=1, display=False)
    week_summaries = hub.run(period="week", year=2024, week=18, top_n=1, display=False)
    month_summaries = hub.run(period="month", year=2024, month=5, top_n=1, display=False)
    custom_summaries = hub.run(
        period="custom",
        start=date(2024, 5, 1),
        end=date(2024, 5, 3),
        top_n=1,
        display=False,
    )
    print(len(day_summaries), len(week_summaries), len(month_summaries), len(custom_summaries))
else:
    print("Additional live examples are disabled. Set RUN_MORE_LIVE_EXAMPLES = True to run them.")

## 12. Async Usage: `await hub.arun(...)`

Jupyter already supports async, so you can use `await hub.arun(...)` directly.

`hub.run(...)` also works in normal notebook usage; PaperHub handles the Jupyter event loop internally.

The next cell is disabled by default because enabling it may trigger another live run.


In [ ]:
RUN_ASYNC_EXAMPLE = False

if RUN_ASYNC_EXAMPLE:
    async_summaries = await hub.arun(**LIVE_RUN_KWARGS, language=LANGUAGE)
    print(f"Async summaries: {len(async_summaries)}")
else:
    print("Async example is disabled. Set RUN_ASYNC_EXAMPLE = True to run it.")

## 13. Cache Check

PaperHub caches:

- Paper metadata
- PDF text
- Model- and language-specific summaries

A second run with the same `arxiv_id`, `model`, and `language` returns from cache when possible. If you change the model or output language, the summary is generated again.


In [ ]:
print("Cache root:", hub.cache.root)
print("SQLite DB:", hub.cache.db_path)
print("PDF cache:", hub.cache.pdf_dir)

if summaries:
    cached = hub.cache.get_summary(summaries[0].arxiv_id, hub.model, LANGUAGE)
    print("Is the first summary cached?:", cached is not None)

## 14. Switching Providers

To use another provider, change `PROVIDER` and `MODEL` near the top and rerun the notebook from top to bottom.

Examples:

```python
hub = PaperHub(provider="openai")     # .env or built-in OpenAI default model
hub = PaperHub(provider="anthropic")  # .env or built-in Anthropic default model
hub = PaperHub(provider="google")     # .env or built-in Google default model

# To explicitly override the model:
hub = PaperHub(provider="openai", model="gpt-5.4-mini")
```

Required environment variable:

- OpenAI: `OPENAI_API_KEY`
- Anthropic: `ANTHROPIC_API_KEY`
- Google: `GOOGLE_API_KEY`

For lower-level usage, `build_llm(model=..., provider=...)` creates only the LLM client. Normal PaperHub usage does not require this.

Provider-specific `.env` model variables:

- `PAPERHUB_ANTHROPIC_MODEL=claude-haiku-4-5-20251001`
- `PAPERHUB_OPENAI_MODEL=gpt-5.4-mini`
- `PAPERHUB_OPENAI_REASONING_EFFORT=xhigh`
- `PAPERHUB_GOOGLE_MODEL=gemini-3-flash-preview`

`PAPERHUB_MODEL` can still be used as a global override. If PaperHub detects that it clearly belongs to another provider, it ignores it and falls back to the selected provider's default.


In [ ]:
from paperhub import build_llm

# This only creates the client object; it does not call the provider.
llm_client = build_llm(model=MODEL, provider=PROVIDER)
print(type(llm_client).__name__, llm_client.provider, llm_client.model)

## 15. Common Errors

- `Missing API key`: run the API key cell in step 3 or fill your `.env` file.
- `model not found`: change `MODEL` to a valid model id from the provider dashboard.
- `No papers found`: HuggingFace Daily Papers may not have data for that date. Change `LIVE_RUN_KWARGS` or `CANDIDATE_RUNS`.
- If PDF text extraction is short, PaperHub tries to continue with the abstract.
- If one paper fails, the whole run does not stop; that paper's `PaperSummary.error` is populated.

Safest first live run:

```python
hub = PaperHub(provider=PROVIDER, model=MODEL, language=LANGUAGE, concurrency=1)
summaries = hub.run(period="day", year=2024, month=5, day=1, top_n=1, display=True)
```
